# MAC Applied Artificial Intelligence · Week 4 Practical

**Google Colab · Student working notebook · 85–90 minutes per task**

This is a **starter notebook**, not a completed solution. Execute the cells **in order**. Keep your real computed outputs visible. Each task includes **student-dependent data** and intentionally unfinished decision cells.

**Rules:** Use your own student ID; don't change it after generating your dataset. Do not upload patient data. Submit **screenshots of outputs**, your **run token**, and a short interpretation in the separate Practical Submission app. Explanations without matching Colab evidence are incomplete. There are no downloadable external datasets.

**Academic integrity:** A generated notebook can still be solved using AI tools. The activity therefore requires live, individualized runs and interpretation, not just filled code.

## Task 2 — Multimodal Data: Image + Text + Sensor Patterns
Concepts consolidated from original Labs 6–10: multimodal transformations, image corruption, image duplicates, noise, and smoothing. The built-in images come from `sklearn` digits (not clinical images) and texts are supplied in-cell, so no downloads are needed.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, accuracy_score
import hashlib

student_id = input('Enter your Student ID: ').strip()
assert len(student_id) >= 4 and student_id.replace('-','').replace('_','').isalnum(), 'Enter a valid student ID.'
seed = int(hashlib.sha256(student_id.encode()).hexdigest()[:8],16)
rng = np.random.default_rng(seed)
run_token = hashlib.sha256((student_id+'|MAC-W4|'+str(seed)).encode()).hexdigest()[:12].upper()
print('STUDENT ID:',student_id, '| RUN TOKEN:',run_token)
print('Take screenshot E0 showing your run token and student ID.')

### Activity M1 · Image quality (Easy · 20 min)
Use the bundled `sklearn.datasets.load_digits` (8×8 image arrays). For a reproducible student-dependent image, compare original, noisy, blurred and repeated samples.

In [ ]:
from sklearn.datasets import load_digits
from scipy.ndimage import gaussian_filter
from PIL import Image
D=load_digits();chosen=int(rng.integers(0,len(D.images)))
original=D.images[chosen].astype(float)
noisy=np.clip(original+rng.normal(0,1.8,original.shape),0,16)
blurred=gaussian_filter(original,sigma=1.0)
duplicate=original.copy()
fig,axes=plt.subplots(1,4,figsize=(10,3))
for ax,img,title in zip(axes,[original,noisy,blurred,duplicate],['I1 original','I2 noisy','I3 blurred','I4 duplicate']):
    ax.imshow(img,cmap='gray',vmin=0,vmax=16);ax.set_title(title);ax.axis('off')
plt.show()
print('Image ID:',chosen,'Label:',D.target[chosen]);print('Exact duplicate:',np.array_equal(original,duplicate))

In [ ]:
# TODO M1: compute and show MSE of noisy and blurred image relative to original.
# Required variables: mse_noisy, mse_blurred
mse_noisy = None
mse_blurred = None
# Add implementation
if mse_noisy is not None and mse_blurred is not None:
    print('M1 noisy MSE:',round(float(mse_noisy),3),'blurred MSE:',round(float(mse_blurred),3))

### Activity M2 · Label noise and split contamination (Medium · 20 min)
Use the given copied image to explain duplicate leakage; examine whether two identical arrays placed in train/test would invalidate an independent assessment. Complete a programmatic duplicate check.

In [ ]:
image_collection=[original, noisy, blurred, duplicate]
print('Displayed sample arrays:',len(image_collection))
# TODO M2: Detect every pair (i,j) with exactly equal arrays using np.array_equal.
# Save as duplicate_pairs and print it. Image numbers are zero-based.
duplicate_pairs=[]
# Add implementation
print('Duplicate image index pairs:',duplicate_pairs)

### Activity M3 · Text cleaning & negation (Medium · 20 min)
No pretrained embeddings or live inference is required. You will compare simple **TF–IDF cosine similarity**, which is lexical similarity rather than a claim of full semantic understanding. Investigate the harm of discarding negation.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
texts=['The patient is not in pain','The patient is in pain','Patient reports no pain','Wound dressing was changed']
order=rng.permutation(len(texts));display(pd.DataFrame({'id':[f'T{x+1}' for x in range(4)],'text':[texts[i] for i in order]}))
vec=TfidfVectorizer(ngram_range=(1,2))
emb=vec.fit_transform([texts[i] for i in order]);sim=cosine_similarity(emb)
print('TF–IDF cosine matrix:');display(pd.DataFrame(np.round(sim,2)))

In [ ]:
# TODO M3: make a SECOND TF-IDF representation after removing only the word 'not'
# from each text (case insensitive); do not remove 'no'.
# Show similarity matrix before and after. Briefly explain why it can be misleading.
texts_without_not = None
# Add implementation

### Activity M4 · Sensor smoothing versus signal loss (Challenging · 20 min)
Generated sensor traces are synthetic and are **not clinical EEG**. Verify how stronger smoothing changes peak height and approximation error.

In [ ]:
t=np.linspace(0,2,400)
clean=np.sin(2*np.pi*4*t)+1.25*np.exp(-((t-1.15)/.035)**2)
noise=rng.normal(0,.25,len(t));observed=clean+noise
window=11
smooth=np.convolve(observed,np.ones(window)/window,mode='same')
plt.figure(figsize=(9,3));plt.plot(t,clean,label='Reference');plt.plot(t,observed,label='Noisy',alpha=.5);plt.plot(t,smooth,label='Smoothed');plt.legend();plt.show()
print('Current window:',window,'| smoothed peak:',round(float(smooth.max()),3))

In [ ]:
# TODO M4: repeat smoothing for window sizes 5, 21, 51.
# Build a table with window, peak amplitude and MSE relative to clean.
# Plot all three resulting traces. Decide whether the smoothest is best.
results = []
# Add implementation

### Task 2 evidence & reflection
Screenshot the chosen image index, original/noisy/blurred comparison, numerical MSE, exact-duplicate check, TF–IDF similarity before and after removing `not`, and the smoothing comparison. Your explanation must cite numeric results and a dataset sample associated with your student-specific token.

In [ ]:
print('TASK 2 EVIDENCE | ID:',student_id,'| TOKEN:',run_token)
print('Image ID:',chosen)
print('M1:',(mse_noisy,mse_blurred))
print('M2 duplicate pairs:',duplicate_pairs)
print('M3 completed:',texts_without_not is not None)
print('M4 rows:',len(results))